In [9]:
# TRAINING MODULE

# Data Preparation

import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline
from statsmodels.tsa.holtwinters import ExponentialSmoothing

# Load and prepare data
df = pd.read_csv("cleaned_unemployment_data.csv")

df = df.sort_values(
    ["Gender", "Year", "Quarter_Number"]
).reset_index(drop=True)

# Time index per gender
df["t"] = df.groupby("Gender").cumcount()

test_size = 10

linear_models = {}
poly_models = {}
holt_models = {}

test_data = {}

print("Data prepared successfully.")

Data prepared successfully.


In [10]:
# Model Training

for gender in sorted(df["Gender"].unique()):

    gdf = (
        df[df["Gender"] == gender]
        .copy()
        .sort_values(["Year", "Quarter_Number"])
        .reset_index(drop=True)
    )

    X_gender = gdf[["t"]].values
    y_gender = gdf["Unemployment_Rate"].values

    # Safety check
    if len(y_gender) <= test_size:
        print(f"Skipping {gender}: not enough data.")
        continue

    split_idx = len(y_gender) - test_size

    X_train = X_gender[:split_idx]
    X_test = X_gender[split_idx:]

    y_train = y_gender[:split_idx]
    y_test = y_gender[split_idx:]

    # Save test sets and quarters
    test_data[gender] = {
        "X_test": X_test,
        "y_test": y_test,
        "quarters": gdf["Quarter_Number"].iloc[split_idx:]
    }

    # 1) Linear Regression
    linear_model = LinearRegression()
    linear_model.fit(X_train, y_train)

    linear_models[gender] = linear_model

    # 2) Polynomial Regression
    poly_model = make_pipeline(
        PolynomialFeatures(
            degree=2,
            include_bias=False
        ),
        LinearRegression()
    )

    poly_model.fit(X_train, y_train)

    poly_models[gender] = poly_model

    # 3) Holt-Winters
    hw = ExponentialSmoothing(
        y_train,
        trend="add",
        seasonal="add",
        seasonal_periods=4
    ).fit(optimized=True)

    holt_models[gender] = hw

print("Linear Regression trained successfully for both genders.")
print("Polynomial Regression trained successfully for both genders.")
print("Holt-Winters trained successfully for both genders.")

Linear Regression trained successfully for both genders.
Polynomial Regression trained successfully for both genders.
Holt-Winters trained successfully for both genders.
